# 5. 아파트 매매 실거래가 다운로드

**국토교통부 아파트 매매 실거래가 상세 자료 API**를 이용해  
2019~2023년, 248개 시군구의 아파트 거래 데이터를 수집

In [9]:
import requests
import pandas as pd
import time
import os
import xml.etree.ElementTree as ET

# ⚠️ 여기에 data.go.kr의 '일반 인증키 (Encoding)' 버전을 붙여넣으세요!
# (마이페이지 → 오픈API → 활용신청 현황 → API 클릭 → Encoding 탭)
API_KEY_ENCODED = "4a29f7168b5c2e0fc38b722c1257826cf9a1098053c86d947c1a6f6973af1aab"

# API 엔드포인트 (PDF 기술문서 기준 정확한 URL)
BASE_URL = "https://apis.data.go.kr/1613000/RTMSDataSvcAptTradeDev/getRTMSDataSvcAptTradeDev"

print("설정 완료")
print(f"Base URL: {BASE_URL}")

설정 완료
Base URL: https://apis.data.go.kr/1613000/RTMSDataSvcAptTradeDev/getRTMSDataSvcAptTradeDev


## 셀 1-1. API 연결 테스트

In [ ]:
# 서울 종로구(법정동 코드 11110), 2023년 1월로 테스트
test_url = (
    f"{BASE_URL}"
    f"?serviceKey={API_KEY_ENCODED}" # 인코딩된 키를 URL에 직접 삽입
    f"&pageNo=1"
    f"&numOfRows=3"
    f"&LAWD_CD=11110" # 법정동 코드 (KOSIS 코드 11010 아님!)
    f"&DEAL_YMD=202301"
)

res = requests.get(test_url, timeout=15)
print("상태 코드:", res.status_code)
print("응답 원문 (앞 800자):")
print(res.text[:800])

# 200이고 <resultCode>000</resultCode> 가 보이면 정상

상태 코드: 200
응답 원문 (앞 800자):
<?xml version="1.0" encoding="utf-8" standalone="yes"?><response><header><resultCode>000</resultCode><resultMsg>OK</resultMsg></header><body><items><item><aptDong>205</aptDong><aptNm>경희궁자이(2단지)</aptNm><aptSeq>11110-2445</aptSeq><bonbun>0199</bonbun><bubun>0000</bubun><buildYear>2017</buildYear><buyerGbn> </buyerGbn><cdealDay> </cdealDay><cdealType> </cdealType><dealAmount>169,000</dealAmount><dealDay>27</dealDay><dealMonth>1</dealMonth><dealYear>2023</dealYear><dealingGbn>중개거래</dealingGbn><estateAgentSggNm>서울 종로구</estateAgentSggNm><excluUseAr>84.8359</excluUseAr><floor>12</floor><jibun>199</jibun><landCd>1</landCd><landLeaseholdGbn>N</landLeaseholdGbn><rgstDate>23.03.16</rgstDate><roadNm>송월길</roadNm><roadNmBonbun>00099</roadNmBonbun><roadNmBubun>00000</roadNmBubun><roadNmCd>4100192</roadNm


## 셀 2. 시군구 코드 불러오기

**코드 헷갈리는 부분 정리:**
| 코드 종류 | 예시 (종로구) | 어디서 쓰이나 |
|-----------|--------------|---------------|
| KOSIS 코드 | `11010` | cluster_aging_merged.csv의 SGG_CODE |
| 법정동 코드 | `11110` | **API의 LAWD_CD** ← 이걸 써야 함 |

`legal_to_kosis_sgg.csv`에서 `legal5` 컬럼이 API에 쓸 법정동 코드.


In [11]:
# 클러스터 + 고령화율 합산 데이터 (248개 시군구)
df_cluster = pd.read_csv('cluster_aging_merged.csv')

# 법정동코드 ↔ KOSIS코드 매핑 테이블
df_legal = pd.read_csv('legal_to_kosis_sgg.csv')

# KOSIS 코드 → 법정동 5자리 코드 딕셔너리 생성
# legal_to_kosis_sgg.csv 에는:
#   legal5 = API에서 쓸 LAWD_CD (법정동 코드 앞 5자리)
#   KOSIS_SGG_CODE = 우리 분석에서 써온 코드
def normalize_kosis_code(series):
    return pd.to_numeric(series, errors='coerce').astype('Int64').astype(str).str.zfill(5)

kosis_to_legal = dict(
    zip(
        normalize_kosis_code(df_legal['KOSIS_SGG_CODE']),
        df_legal['legal5'].astype(str).str.zfill(5)
    )
)

print(f"매핑 테이블 크기: {len(kosis_to_legal)}개 시군구")
print("예시 (KOSIS → 법정동):")
for kosis, legal in list(kosis_to_legal.items())[:5]:
    print(f"  KOSIS {kosis} → 법정동 {legal}")

# cluster_aging_merged 의 KOSIS 코드들을 법정동 코드로 변환
sgg_kosis_codes = normalize_kosis_code(df_cluster['SGG_CODE']).tolist()

# (KOSIS코드, 법정동코드) 쌍 목록 - 매핑 안 되는 것은 제외
sgg_pairs = []
skipped = []
for kosis in sgg_kosis_codes:
    legal = kosis_to_legal.get(kosis)
    if legal:
        sgg_pairs.append((kosis, legal))
    else:
        skipped.append(kosis)

print(f"\nAPI 호출 가능: {len(sgg_pairs)}개 시군구")
if skipped:
    print(f"매핑 없어서 제외: {skipped}")

매핑 테이블 크기: 261개 시군구
예시 (KOSIS → 법정동):
  KOSIS 11010 → 법정동 11110
  KOSIS 11020 → 법정동 11140
  KOSIS 11030 → 법정동 11170
  KOSIS 11040 → 법정동 11200
  KOSIS 11050 → 법정동 11215

API 호출 가능: 248개 시군구


## 셀 3. 년월 목록 생성

In [12]:
# 2019년 1월 ~ 2023년 12월 = 60개월
year_months = []
for year in range(2019, 2024):
    for month in range(1, 13):
        year_months.append(f"{year}{month:02d}")  # 예: '201901', '202312'

total_calls = len(sgg_pairs) * len(year_months)
print(f"총 년월 수: {len(year_months)}개 (2019.01 ~ 2023.12)")
print(f"총 API 호출 횟수: {len(sgg_pairs)} × {len(year_months)} = {total_calls:,}번")
print(f"예상 소요 시간 (0.1초 간격): 약 {total_calls * 0.1 / 60:.0f}분")

총 년월 수: 60개 (2019.01 ~ 2023.12)
총 API 호출 횟수: 248 × 60 = 14,880번
예상 소요 시간 (0.1초 간격): 약 25분


## 셀 4. API 호출 함수 정의

In [13]:
def fetch_apt_trade(lawd_cd_legal, deal_ymd, max_rows=1000):
    # URL을 직접 조립 - 인코딩된 키를 그대로 붙임
    url = (
        f"{BASE_URL}"
        f"?serviceKey={API_KEY_ENCODED}"   # 인코딩 키 직접 삽입
        f"&pageNo=1"
        f"&numOfRows={max_rows}"
        f"&LAWD_CD={lawd_cd_legal}"        # 법정동 코드!
        f"&DEAL_YMD={deal_ymd}"
    )
    
    try:
        res = requests.get(url, timeout=15)
        
        # HTTP 에러 처리 (401, 500 등)
        if res.status_code != 200:
            return None, f"HTTP {res.status_code}: {res.text[:80]}"
        
        # XML 파싱
        root = ET.fromstring(res.text)
        
        # API 레벨 에러 코드 확인 (정상은 '000')
        result_code = root.find('.//resultCode')
        result_msg  = root.find('.//resultMsg')
        code = result_code.text if result_code is not None else 'NONE'
        msg  = result_msg.text  if result_msg  is not None else ''
        
        if code != '000':   # ← '00' 아니라 '000' !
            return [], f"API code={code}: {msg}"
        
        # 아이템 파싱
        items = []
        for item in root.findall('.//item'):
            row = {child.tag: child.text for child in item}
            items.append(row)
        
        return items, None   # (데이터 리스트, 에러없음)
    
    except ET.ParseError:
        return None, f"XML 파싱 실패"
    except Exception as e:
        return None, str(e)

print("함수 정의 완료")

함수 정의 완료


## 셀 4-1. 함수 테스트 (종로구 2023년 1월)

In [14]:
# 종로구(법정동 11110)로 테스트
items, err = fetch_apt_trade('11110', '202301')

if err and not items:
    print(f"에러: {err}")
elif not items:
    print("데이터 없음 (해당 월 거래 0건)")
else:
    print(f"조회 성공: {len(items)}건")
    print("\n첫 번째 데이터:")
    for k, v in items[0].items():
        print(f"  {k}: {v}")

조회 성공: 7건

첫 번째 데이터:
  aptDong: 205
  aptNm: 경희궁자이(2단지)
  aptSeq: 11110-2445
  bonbun: 0199
  bubun: 0000
  buildYear: 2017
  buyerGbn:  
  cdealDay:  
  cdealType:  
  dealAmount: 169,000
  dealDay: 27
  dealMonth: 1
  dealYear: 2023
  dealingGbn: 중개거래
  estateAgentSggNm: 서울 종로구
  excluUseAr: 84.8359
  floor: 12
  jibun: 199
  landCd: 1
  landLeaseholdGbn: N
  rgstDate: 23.03.16
  roadNm: 송월길
  roadNmBonbun: 00099
  roadNmBubun: 00000
  roadNmCd: 4100192
  roadNmSeq: 01
  roadNmSggCd: 11110
  roadNmbCd:  
  sggCd: 11110
  slerGbn:  
  umdCd: 17900
  umdNm: 홍파동


## 셀 5. 전체 데이터 다운로드

In [15]:
# 저장 폴더 생성
save_dir = '아파트실거래가'
os.makedirs(save_dir, exist_ok=True)

all_data = []     # 전체 수집 데이터
error_log = []    # 실패한 요청 기록
total = len(sgg_pairs) * len(year_months)
count = 0

for kosis_cd, legal_cd in sgg_pairs:
    for ym in year_months:
        count += 1
        
        # 진행률 출력 (500번마다)
        if count % 500 == 0 or count == 1:
            pct = count / total * 100
            print(f"[{count:5d}/{total}] {pct:.1f}% | {kosis_cd}→{legal_cd} {ym} | 수집: {len(all_data):,}건")
        
        # API 호출 (법정동 코드 사용!)
        items, err = fetch_apt_trade(legal_cd, ym)
        
        if err and items is None:
            # 치명적 에러 (401 등) - 기록하고 계속
            error_log.append({'kosis': kosis_cd, 'legal': legal_cd, 'ym': ym, 'error': err})
        elif items:
            # 각 거래 데이터에 시군구 코드 추가
            for item in items:
                item['KOSIS_SGG_CODE'] = kosis_cd   # 분석용 KOSIS 코드
                item['LEGAL_SGG_CODE'] = legal_cd   # API 호출에 쓴 법정동 코드
            all_data.extend(items)
        
        # API 과부하 방지
        time.sleep(0.1)

# 최종 저장
df_apt = pd.DataFrame(all_data)
save_path = f'{save_dir}/apt_raw_2019_2023.csv'
df_apt.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"\n완료")
print(f"   총 {len(df_apt):,}건 저장 → {save_path}")
print(f"   에러 발생: {len(error_log)}건")

if error_log:
    df_err = pd.DataFrame(error_log)
    df_err.to_csv(f'{save_dir}/error_log.csv', index=False)
    print(f"   에러 로그 → {save_dir}/error_log.csv")

df_apt.head()

[    1/14880] 0.0% | 11010→11110 201901 | 수집: 0건
[  500/14880] 3.4% | 11090→11305 202008 | 수집: 59,264건
[ 1000/14880] 6.7% | 11170→11530 202204 | 수집: 160,042건
[ 1500/14880] 10.1% | 11250→11740 202312 | 수집: 248,502건
[ 2000/14880] 13.4% | 21090→26350 202008 | 수집: 341,959건
[ 2500/14880] 16.8% | 22010→27110 202204 | 수집: 442,449건
[ 3000/14880] 20.2% | 22520→27720 202312 | 수집: 565,789건
[ 3500/14880] 23.5% | 23310→28710 202008 | 수집: 729,830건
[ 4000/14880] 26.9% | 25020→30140 202204 | 수집: 854,634건
[ 4500/14880] 30.2% | 26310→31710 202312 | 수집: 996,312건
[ 5000/14880] 33.6% | 31030→41150 202008 | 수집: 1,126,248건
[ 5500/14880] 37.0% | 31092→41273 202204 | 수집: 1,250,902건
[ 6000/14880] 40.3% | 31150→41390 202312 | 수집: 1,399,722건
[ 6500/14880] 43.7% | 31220→41550 202008 | 수집: 1,536,371건
[ 7000/14880] 47.0% | 31370→41820 202204 | 수집: 1,611,668건
[ 7500/14880] 50.4% | 32070→51230 202312 | 수집: 1,700,227건
[ 8000/14880] 53.8% | 32390→51810 202008 | 수집: 1,709,424건
[ 8500/14880] 57.1% | 33044→43114 202204 | 수

,aptDong,aptNm,aptSeq,bonbun,bubun,buildYear,buyerGbn,cdealDay,cdealType,dealAmount,...,roadNmCd,roadNmSeq,roadNmSggCd,roadNmbCd,sggCd,slerGbn,umdCd,umdNm,KOSIS_SGG_CODE,LEGAL_SGG_CODE
0,,광화문스페이스본(101동~105동),11110-2203,0009,0000,2008,,,,"119,000",...,4100135,03,11110,0,11110,,11500,사직동,11010,11110
1,,벽산블루밍평창힐스,11110-146,0045,0000,2004,,,,"99,000",...,3100023,01,11110,0,11110,,18300,평창동,11010,11110
2,,동대문,11110-30,0328,0017,1966,,,,"29,000",...,3005007,01,11110,0,11110,,17400,창신동,11010,11110
3,,롯데낙천대,11110-71,0072,0000,2001,,,,"57,500",...,3100023,01,11110,0,11110,,18300,평창동,11010,11110
4,,삼성,11110-73,0596,0000,1998,,,,"35,500",...,3100023,01,11110,0,11110,,18300,평창동,11010,11110


## 셀 6. 데이터 미리보기 및 기본 확인

In [16]:
# 다운로드 후 기본 확인
print(f"데이터 크기: {df_apt.shape}")
print(f"\n컬럼 목록:")
print(df_apt.columns.tolist())

print(f"\n거래금액 변환 (문자 '12,000' → 숫자 12000):")
# dealAmount는 '12,000' 형태 → 숫자로 변환
df_apt['dealAmount_만원'] = (
    df_apt['dealAmount']
    .str.replace(',', '', regex=False)   # 쉼표 제거
    .str.strip()                          # 공백 제거
    .astype(float)                        # 숫자 변환
)

print(df_apt['dealAmount_만원'].describe())

print(f"\n시군구별 거래 건수 (상위 10개):")
print(df_apt.groupby('KOSIS_SGG_CODE').size().sort_values(ascending=False).head(10))

데이터 크기: (1984559, 34)

컬럼 목록:
['aptDong', 'aptNm', 'aptSeq', 'bonbun', 'bubun', 'buildYear', 'buyerGbn', 'cdealDay', 'cdealType', 'dealAmount', 'dealDay', 'dealMonth', 'dealYear', 'dealingGbn', 'estateAgentSggNm', 'excluUseAr', 'floor', 'jibun', 'landCd', 'landLeaseholdGbn', 'rgstDate', 'roadNm', 'roadNmBonbun', 'roadNmBubun', 'roadNmCd', 'roadNmSeq', 'roadNmSggCd', 'roadNmbCd', 'sggCd', 'slerGbn', 'umdCd', 'umdNm', 'KOSIS_SGG_CODE', 'LEGAL_SGG_CODE']

거래금액 변환 (문자 '12,000' → 숫자 12000):
count    1.984559e+06
mean     3.927883e+04
std      3.793613e+04
min      5.710000e+02
25%      1.700000e+04
50%      2.900000e+04
75%      4.900000e+04
max      1.800000e+06
Name: dealAmount_만원, dtype: float64

시군구별 거래 건수 (상위 10개):
KOSIS_SGG_CODE
34012    38594
31070    36209
31130    33258
34040    33107
24040    32733
32020    32697
23080    31689
31150    31671
22070    31008
25030    29691
dtype: int64
